# Chapter 1: Deterministic vs. AI Classification

**From *AI Enterprise Architecture***

This notebook compares a traditional rules-based classifier with an LLM-based classifier
on the same customer support ticket classification task. You will see firsthand how these
two approaches differ in:

- **Determinism** -- does the same input always produce the same output?
- **Handling of ambiguous cases** -- what happens when a ticket doesn't fit neatly into one category?
- **Edge cases** -- how does each approach handle unusual inputs?

**Key insight:** Non-deterministic components require fundamentally different architectural thinking
than the deterministic systems enterprise architects are accustomed to designing.

## Setup

Install the required dependencies and configure your API key.

In [ ]:
!pip install -q openai tabulate

In [ ]:
import os
import time
from openai import OpenAI
from tabulate import tabulate

# Set your OpenAI API key as an environment variable before running.
# In Colab: use the Secrets panel (key icon in the left sidebar),
# then uncomment the two lines below:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Set it as an environment variable "
        "or add it via Colab Secrets."
    )

client = OpenAI(api_key=api_key)
print("OpenAI client ready.")

## 1. Define the Test Data

We create 10 customer support tickets that span three categories:
**billing**, **technical**, and **general**. Some tickets are straightforward;
others are deliberately ambiguous to expose differences between the two classifiers.

In [ ]:
TICKETS = [
    # Clear-cut cases
    "I was charged twice for my subscription last month.",                      # billing
    "The app crashes every time I try to upload a photo.",                      # technical
    "What are your business hours?",                                            # general
    "Please refund the $49.99 charge on my credit card.",                       # billing
    "I'm getting a 502 Bad Gateway error on the dashboard.",                    # technical
    # Ambiguous / edge cases
    "I can't log in and I think you're still charging me.",                     # billing + technical
    "Your service is terrible and I want to cancel everything.",                # billing + general
    "How do I export my data before my trial expires?",                         # technical + billing
    "",                                                                         # empty string
    "asdjkfh 1234 !!! ???",                                                    # gibberish
]

CATEGORIES = ["billing", "technical", "general"]

print(f"Loaded {len(TICKETS)} tickets across categories: {CATEGORIES}")

## 2. Rules-Based Classifier

This is the kind of component a traditional enterprise architect might design:
a deterministic function that maps keywords to categories. It is predictable,
testable, and always returns the same result for the same input.

In [ ]:
def classify_rules(ticket: str) -> str:
    """Classify a support ticket using keyword matching.

    Priority order: billing > technical > general (fallback).
    """
    text = ticket.lower()

    billing_keywords = [
        "charge", "refund", "invoice", "billing", "payment",
        "subscription", "cancel", "price", "cost", "credit card",
        "expires", "trial",
    ]
    technical_keywords = [
        "error", "crash", "bug", "login", "log in", "password",
        "upload", "download", "502", "404", "slow", "broken",
        "export", "data",
    ]

    if any(kw in text for kw in billing_keywords):
        return "billing"
    if any(kw in text for kw in technical_keywords):
        return "technical"
    return "general"


# Quick sanity check
for t in TICKETS:
    preview = t[:60] if t else "(empty)"
    print(f"  {classify_rules(t):>10}  |  {preview}")

## 3. LLM-Based Classifier

Now we use GPT to classify the same tickets. Notice that we ask for a single
word from a fixed set of categories -- but the model is free to interpret the
text however it sees fit. We use `temperature=0.0` to reduce randomness, but
even so the results may vary across runs.

In [ ]:
SYSTEM_PROMPT = (
    "You are a support ticket classifier. "
    "Classify the user's message into exactly one category: "
    "billing, technical, or general. "
    "Respond with the single category word only, in lowercase."
)


def classify_llm(ticket: str) -> str:
    """Classify a support ticket using an LLM."""
    if not ticket.strip():
        # We still send it -- let's see what the model does with empty input
        user_content = "(empty message)"
    else:
        user_content = ticket

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        max_tokens=10,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
    )
    raw = response.choices[0].message.content.strip().lower()
    # Normalise: if the model returns something unexpected, flag it
    if raw not in CATEGORIES:
        return f"UNEXPECTED: {raw}"
    return raw


# Single test
print("LLM test:", classify_llm(TICKETS[0]))

## 4. Run Both Classifiers Multiple Times

We run each classifier on every ticket **3 times**. The rules-based classifier
will always produce the same results. The LLM classifier *might* vary, even with
`temperature=0.0`, because the API is not guaranteed to be perfectly deterministic.

In [ ]:
NUM_RUNS = 3

rules_results = []   # list of lists: [run][ticket_idx]
llm_results = []     # same shape

for run in range(NUM_RUNS):
    print(f"--- Run {run + 1}/{NUM_RUNS} ---")
    rules_run = [classify_rules(t) for t in TICKETS]
    rules_results.append(rules_run)

    llm_run = []
    for t in TICKETS:
        result = classify_llm(t)
        llm_run.append(result)
        time.sleep(0.3)  # polite rate limiting
    llm_results.append(llm_run)

print("\nAll runs complete.")

## 5. Compare Results

We build a comparison table that shows, for each ticket:
- The rules-based result (all 3 runs -- expected to be identical)
- The LLM-based result for each run
- Whether the LLM results were consistent across runs

In [ ]:
table_rows = []

for i, ticket in enumerate(TICKETS):
    preview = ticket[:45] + "..." if len(ticket) > 45 else ticket
    if not preview:
        preview = "(empty)"

    rules_vals = [rules_results[r][i] for r in range(NUM_RUNS)]
    llm_vals = [llm_results[r][i] for r in range(NUM_RUNS)]

    rules_consistent = "Yes" if len(set(rules_vals)) == 1 else "NO"
    llm_consistent = "Yes" if len(set(llm_vals)) == 1 else "NO"

    table_rows.append([
        i + 1,
        preview,
        rules_vals[0],
        rules_consistent,
        " / ".join(llm_vals),
        llm_consistent,
    ])

headers = ["#", "Ticket", "Rules", "Rules Consistent?", "LLM (3 runs)", "LLM Consistent?"]
print(tabulate(table_rows, headers=headers, tablefmt="github"))

## 6. Analysis: Where the Two Approaches Diverge

Let's highlight the specific cases where the rules-based and LLM classifiers disagree.

In [ ]:
print("Cases where rules-based and LLM classifiers DISAGREE:\n")

disagreements = 0
for i, ticket in enumerate(TICKETS):
    rules_label = rules_results[0][i]
    llm_labels = set(llm_results[r][i] for r in range(NUM_RUNS))

    if rules_label not in llm_labels:
        disagreements += 1
        preview = ticket[:70] if ticket else "(empty)"
        print(f"  Ticket {i+1}: \"{preview}\"")
        print(f"    Rules  -> {rules_label}")
        print(f"    LLM    -> {llm_labels}")
        print()

if disagreements == 0:
    print("  (No disagreements in this run -- try adding more ambiguous tickets!)")
else:
    print(f"Total disagreements: {disagreements} out of {len(TICKETS)} tickets.")

## Key Takeaways for Enterprise Architects

| Aspect | Rules-Based | LLM-Based |
|--------|------------|----------|
| **Determinism** | Always the same output for same input | May vary across calls, even with temperature=0 |
| **Ambiguity handling** | Falls through to first matching rule | Interprets meaning and context |
| **Edge cases** | Predictable failures (empty input -> default) | Unpredictable but often reasonable |
| **Maintenance** | Must manually update keyword lists | Adapts to new phrasing automatically |
| **Testability** | Simple unit tests | Requires evaluation frameworks |
| **Latency** | Microseconds | Hundreds of milliseconds |

**The architectural implication:** When you replace a deterministic component with an
AI component, you are not just swapping an implementation -- you are changing the
fundamental contract of that component. Your architecture must account for
non-determinism, latency, and the need for continuous evaluation.